In [2]:
import numpy as np

# 兼容旧版本 sklearn / 其它库用的 np.float / np.int
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int


In [2]:
# Faster version for Jupyter (restarts clean): uses lighter CV and smaller XGB to avoid timeouts.
import json
import numpy as np
import numpy as np
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int

import pandas as pd
from sklearn.calibration import calibration_curve
import matplotlib.gridspec as gridspec
import shap

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, brier_score_loss,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def hr(title: str):
    print("\n" + title)
    print("-" * len(title))

def full_metrics(y_true, y_prob, threshold=0.5, prefix=""):
    y_hat = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_hat)
    try: auroc = roc_auc_score(y_true, y_prob)
    except Exception: auroc = float("nan")
    try: auprc = average_precision_score(y_true, y_prob)
    except Exception: auprc = float("nan")
    prec = precision_score(y_true, y_hat, zero_division=0)
    rec = recall_score(y_true, y_hat, zero_division=0)
    f1 = f1_score(y_true, y_hat, zero_division=0)
    brier = brier_score_loss(y_true, y_prob)
    print(f"[{prefix}] Acc={acc:.4f} | AUROC={auroc:.4f} | AUPRC={auprc:.4f} | "
          f"P={prec:.4f} | R={rec:.4f} | F1={f1:.4f} | Brier={brier:.4f}")
    print(classification_report(y_true, y_hat, digits=4))
    return {"accuracy": float(acc), "auroc": float(auroc), "auprc": float(auprc),
            "precision": float(prec), "recall": float(rec), "f1": float(f1),
            "brier": float(brier), "threshold": float(threshold)}

def youden_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return float(thr[np.argmax(j)])

def plot_calibration_with_hist(
    y_true,
    y_prob,
    threshold,
    strategy="quantile",
    n_bins=10,
    title="Calibration (Reliability) + Risk Histogram",
    out_path=None,
):

    prob_true, prob_pred = calibration_curve(
        y_true, y_prob, n_bins=n_bins, strategy=strategy
    )

    fig = plt.figure(figsize=(6, 6))
    gs = gridspec.GridSpec(
        2, 1, height_ratios=[2, 1], hspace=0.05 
    )
    ax_cal = fig.add_subplot(gs[0])

    # 完美校准线
    ax_cal.plot([0, 1], [0, 1], linestyle="--", label="Perfect")

    # 实际校准点
    ax_cal.plot(prob_pred, prob_true, marker="o", linestyle="-", label="Observed")

    # Youden 阈值竖线
    ax_cal.axvline(threshold, linestyle=":", label=f"Youden thr={threshold:.2f}")

    ax_cal.set_ylabel("Observed event rate")
    ax_cal.set_title(title)
    ax_cal.set_xlim(0.0, 1.0)
    ax_cal.set_ylim(0.0, 1.0)
    ax_cal.legend(loc="lower right")
    ax_cal.tick_params(labelbottom=False)

    ax_hist = fig.add_subplot(gs[1], sharex=ax_cal)
    ax_hist.hist(y_prob, bins=20)
    ax_hist.axvline(threshold, linestyle=":", label="Youden threshold")
    ax_hist.set_xlabel("Predicted risk")
    ax_hist.set_ylabel("Count")
    ax_hist.set_xlim(0.0, 1.0)

    plt.tight_layout()

    if out_path is not None:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


def save_confusion(y_true, y_prob, threshold, out_path):
    y_hat = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_hat)
    fig = plt.figure(figsize=(4, 4))
    plt.imshow(cm)  # default colormap
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    plt.tight_layout()
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=160)
    plt.close(fig)

# Load data
csv_path = "D:/project/icu_outputs11/full_selected.csv"
df = pd.read_csv(csv_path)
print(f"✅ Loaded: {Path(csv_path).name} | shape={df.shape}")

# Detect columns
ID_CANDIDATES = ["icustay_id", "subject_id", "hadm_id", "RecordID", "patient_id"]
TIME_CANDIDATES = ["event_time", "window_end_time", "end_time", "charttime", "timestamp"]
TARGET_CANDIDATES = ["In-hospital_death", "In_hospital_death", "hospital_death", "mortality", "label", "y"]
def find_first(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None
cols = df.columns.tolist()
ID_COL = find_first(cols, ID_CANDIDATES)
TIME_COL = find_first(cols, TIME_CANDIDATES)
TARGET_COL = find_first(cols, TARGET_CANDIDATES)
assert TARGET_COL is not None

# Sentinels to NaN
SENTINELS = {-1, -9, 999, 9999}
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        df.loc[df[c].isin(SENTINELS), c] = np.nan

# De-dup
if ID_COL is not None:
    before = df.shape[0]
    if TIME_COL is not None:
        df = df.sort_values([ID_COL, TIME_COL]).groupby(ID_COL, as_index=False, sort=False).tail(1)
    else:
        df = df.drop_duplicates(subset=[ID_COL], keep="last")
    after = df.shape[0]
    print(f"🔎 De-dup by {ID_COL}: {before} → {after} rows; unique IDs: {df[ID_COL].nunique()}")

# Features
ignore_cols = {TARGET_COL}
if ID_COL: ignore_cols.add(ID_COL)
if TIME_COL: ignore_cols.add(TIME_COL)
num_cols = [c for c in df.columns if c not in ignore_cols and pd.api.types.is_numeric_dtype(df[c])]
print(f"🧠 Using {len(num_cols)} numeric features")

X = df[num_cols].values
y = df[TARGET_COL].values.astype(int)

# Splits
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.10, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=1/9, random_state=RANDOM_STATE, stratify=y_trainval
)
def cnt(yv): 
    return (int(np.sum(yv==0)), int(np.sum(yv==1)))
n0,n1 = cnt(y_train); print(f"🔢 [train] neg={n0}, pos={n1}, pos_rate={n1/(n0+n1):.3f}")
n0,n1 = cnt(y_val);   print(f"🔢 [valid] neg={n0}, pos={n1}, pos_rate={n1/(n0+n1):.3f}")
n0,n1 = cnt(y_test);  print(f"🔢 [test ] neg={n0}, pos={n1}, pos_rate={n1/(n0+n1):.3f}")

# Pipelines
logit = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(solver="lbfgs", max_iter=200, random_state=RANDOM_STATE))
])

neg, pos = np.sum(y_train==0), np.sum(y_train==1)
scale_pos_weight = float((neg / max(pos, 1)) if pos > 0 else 1.0)
print(f"🌲 XGB scale_pos_weight = {scale_pos_weight:.2f}")

xgb = XGBClassifier(
    n_estimators=500,          # smaller to speed up
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric="logloss"
)

# CV (lighter): 10-fold for Logit only (XGB CV skipped to avoid timeouts)
hr("10-fold Cross-Validation on TRAIN (Logistic only)")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_auc_log = cross_val_score(logit, X_train, y_train, cv=skf, scoring="roc_auc")
print(f"[Logit] 10-fold CV AUROC: mean={cv_auc_log.mean():.4f} ± {cv_auc_log.std():.4f}")

# Fit + Eval
out_dir = Path("D:/project/data"); out_dir.mkdir(exist_ok=True, parents=True)

hr("Logistic Regression (Val threshold -> Test metrics)")
logit.fit(X_train, y_train)
val_prob_log = logit.predict_proba(X_val)[:, 1]
th_log = youden_threshold(y_val, val_prob_log)
print(f"[Logit] chosen threshold (Youden@VAL) = {th_log:.3f}")

test_prob_log = logit.predict_proba(X_test)[:, 1]
m_log = full_metrics(y_test, test_prob_log, threshold=th_log, prefix="Logit")
save_confusion(y_test, test_prob_log, th_log, out_dir / "confusion_matrix_logit.png")

# Calibration + risk histogram for Logistic Regression (on TEST)
calib_log_path = out_dir / "calibration_logit_test.png"
plot_calibration_with_hist(
    y_true=y_test,
    y_prob=test_prob_log,
    threshold=th_log,
    strategy="quantile", 
    n_bins=10,
    title="Logistic Regression Calibration (Test)",
    out_path=calib_log_path,
)
print(f"- Calibration (Logit): {calib_log_path}")

hr("XGBoost (Val threshold -> Test metrics)")
xgb.fit(X_train, y_train, verbose=False)
val_prob_xgb = xgb.predict_proba(X_val)[:, 1]
th_xgb = youden_threshold(y_val, val_prob_xgb)
print(f"[XGB ] chosen threshold (Youden@VAL) = {th_xgb:.3f}")

test_prob_xgb = xgb.predict_proba(X_test)[:, 1]
m_xgb = full_metrics(y_test, test_prob_xgb, threshold=th_xgb, prefix="XGBoost")
save_confusion(y_test, test_prob_xgb, th_xgb, out_dir / "confusion_matrix_xgb.png")


# 为了速度，随机采样一部分训练样本来画 SHAP
n_shap = min(2000, X_train.shape[0])
idx_shap = np.random.choice(X_train.shape[0], size=n_shap, replace=False)
X_train_df = pd.DataFrame(X_train, columns=num_cols)
X_shap = X_train_df.iloc[idx_shap].copy()

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap)

fig = plt.figure(figsize=(8, 6))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=num_cols,
    show=False
)
plt.tight_layout()
shap_path = out_dir / "xgb_shap_summary.png"
plt.savefig(shap_path, dpi=160, bbox_inches="tight")
plt.close(fig)

print(f"- SHAP summary (XGB): {shap_path}")


# Save
pred_df = pd.DataFrame({"y_test": y_test, "logit_prob": test_prob_log, "xgb_prob": test_prob_xgb})
pred_path = out_dir / "test_predictions.csv"; pred_df.to_csv(pred_path, index=False)
metrics = {"cv": {"logit": {"auc_mean": float(cv_auc_log.mean()), "auc_std": float(cv_auc_log.std())}},
           "test": {"logit": m_log, "xgb": m_xgb},
           "meta": {"n_total": int(df.shape[0]), "n_features": int(len(num_cols)),
                    "target": TARGET_COL, "id_col": ID_COL, "time_col": TIME_COL}}
metrics_path = out_dir / "metrics_summary.json"; 
with open(metrics_path, "w") as f: json.dump(metrics, f, indent=2)

print("\n💾 Outputs saved:")
print(f"- Predictions:          {pred_path}")
print(f"- Metrics:              {metrics_path}")
print(f"- Confusion (Logit):    {out_dir / 'confusion_matrix_logit.png'}")
print(f"- Confusion (XGB):      {out_dir / 'confusion_matrix_xgb.png'}")
print(f"- Calibration (Logit):  {out_dir / 'calibration_logit_test.png'}")
print(f"- SHAP summary (XGB):   {out_dir / 'xgb_shap_summary.png'}")


`np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
`np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations


✅ Loaded: full_selected.csv | shape=(4000, 49)
🔎 De-dup by RecordID: 4000 → 4000 rows; unique IDs: 4000
🧠 Using 47 numeric features
🔢 [train] neg=2746, pos=454, pos_rate=0.142
🔢 [valid] neg=343, pos=57, pos_rate=0.142
🔢 [test ] neg=343, pos=57, pos_rate=0.142
🌲 XGB scale_pos_weight = 6.05

10-fold Cross-Validation on TRAIN (Logistic only)
-------------------------------------------------
[Logit] 10-fold CV AUROC: mean=0.8511 ± 0.0317

Logistic Regression (Val threshold -> Test metrics)
---------------------------------------------------
[Logit] chosen threshold (Youden@VAL) = 0.124
[Logit] Acc=0.7825 | AUROC=0.8715 | AUPRC=0.6447 | P=0.3790 | R=0.8246 | F1=0.5193 | Brier=0.0794
              precision    recall  f1-score   support

           0     0.9638    0.7755    0.8595       343
           1     0.3790    0.8246    0.5193        57

    accuracy                         0.7825       400
   macro avg     0.6714    0.8000    0.6894       400
weighted avg     0.8804    0.7825    0.81

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.


- Calibration (Logit): D:\project\data\calibration_logit_test.png

XGBoost (Val threshold -> Test metrics)
---------------------------------------


[12:34:33] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.



[XGB ] chosen threshold (Youden@VAL) = 0.177
[XGBoost] Acc=0.7725 | AUROC=0.8652 | AUPRC=0.5746 | P=0.3672 | R=0.8246 | F1=0.5081 | Brier=0.0884
              precision    recall  f1-score   support

           0     0.9632    0.7638    0.8520       343
           1     0.3672    0.8246    0.5081        57

    accuracy                         0.7725       400
   macro avg     0.6652    0.7942    0.6801       400
weighted avg     0.8783    0.7725    0.8030       400

- SHAP summary (XGB): D:\project\data\xgb_shap_summary.png

💾 Outputs saved:
- Predictions:          D:\project\data\test_predictions.csv
- Metrics:              D:\project\data\metrics_summary.json
- Confusion (Logit):    D:\project\data\confusion_matrix_logit.png
- Confusion (XGB):      D:\project\data\confusion_matrix_xgb.png
- Calibration (Logit):  D:\project\data\calibration_logit_test.png
- SHAP summary (XGB):   D:\project\data\xgb_shap_summary.png


  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4


ERROR: Could not install packages due to an EnvironmentError: [WinError 5] 拒绝访问。: 'D:\\a\\Lib\\site-packages\\~umpy\\.libs\\libopenblas64__v0.3.21-gcc_10_3_0.dll'
Consider using the `--user` option or check the permissions.



SyntaxError: invalid character in identifier (<ipython-input-6-50b9260b8f21>, line 1)